In [1]:
!pip install qiskit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 21.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [qiskit]


In [4]:
from qiskit import QuantumCircuit
from qiskit.compiler import transpile
from qiskit.dagcircuit import DAGCircuit
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit import qasm2


# 1. Создаем тестовую "грязную" схему, прямо как в соревновании
# Представим, что мы прочитали этот текст из файла OpenQASM
qasm_code = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];
t q[0];    // Биграмма из двух T: должна превратиться в S

t q[1];
tdg q[1];  // Биграмма T и T-dagger: должны полностью уничтожиться
"""

# Загружаем этот текст в объект квантовой схемы Python
circuit = QuantumCircuit.from_qasm_str(qasm_code)
print("--- ИСХОДНАЯ СХЕМА (До оптимизации) ---")
print(qasm2.dumps(circuit))


# 2. Переводим схему в Направленный Ациклический Граф (DAG)
# Это нужно, чтобы вентили распределились по "дорожкам" кубитов
dag = circuit_to_dag(circuit)

# 3. Логика оптимизатора: проходим по графу и ищем биграммы
def optimize_t_gates(dag: DAGCircuit) -> DAGCircuit:
    # Метод nodes_on_wire_and_nodes_on_edges позволяет идти строго по линии каждого кубита
    for wire in dag.wires:
        # Получаем последовательность всех вентилей на конкретном кубите
        nodes = list(dag.nodes_on_wire(wire))

        i = 0
        while i < len(nodes) - 1:
            node1 = nodes[i]
            node2 = nodes[i+1]

            # Проверяем, являются ли два соседних узла квантовыми гейтами
            if node1.type == 'op' and node2.type == 'op':

                # Сценарий А: Нашли [T] + [T] -> заменяем на [S]
                if node1.name == 't' and node2.name == 't':
                    # Удаляем старые узлы T из графа
                    dag.remove_op_node(node1)
                    dag.remove_op_node(node2)
                    # На их место вставляем один дешевый гейт S
                    # (В реальном коде турнира здесь также прописывается точное место вставки)

                # Сценарий Б: Нашли [T] + [T-dagger] -> полностью удаляем
                elif (node1.name == 't' and node2.name == 'tdg') or (node1.name == 'tdg' and node2.name == 't'):
                    dag.remove_op_node(node1)
                    dag.remove_op_node(node2)

            i += 1
    return dag

# Применяем наш алгоритм (в Qiskit есть встроенные аналоги, но этот написан вручную для наглядности)
# Для полной и глубокой оптимизации в Qiskit используют Transpiler Passes:
optimized_circuit = transpile(circuit, optimization_level=1) # Level 1 как раз делает слияние фаз

print("\n--- ОПТИМИЗИРОВАННАЯ СХЕМА ---")
print(qasm2.dumps(optimized_circuit))




--- ИСХОДНАЯ СХЕМА (До оптимизации) ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
t q[0];
t q[0];
t q[1];
tdg q[1];

--- ОПТИМИЗИРОВАННАЯ СХЕМА ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
u1(pi/2) q[0];


In [5]:
!pip install matplotlib pydot graphviz


In [9]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag
from qiskit.visualization import dag_drawer
import matplotlib.pyplot as plt

# 1. Создаем схему с двухкубитным вентилем CNOT (cx)
# Кубит 0 будет ведущим (Control), Кубит 1 - ведомым (Target)
qasm_with_cnot = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];            // Гейт на ведущем кубите
cx q[0], q[1];     // CNOT: q[0] управляет, q[1] изменяется
t q[1];            // Гейт на ведомом кубите
"""


circuit = QuantumCircuit.from_qasm_str(qasm_with_cnot)

# 2. Превращаем схему в граф (DAG)
dag = circuit_to_dag(circuit)

# 3. Визуализируем граф зависимостей
# Эта функция сгенерирует изображение, где узлы — это гейты,
# а стрелки показывают, как данные текут по ведущим и ведомым ветвям.
dag_drawer(dag, filename="quantum_dag.png")

print("Граф схемы успешно сохранен в файл 'quantum_dag.png'!")


Граф схемы успешно сохранен в файл 'quantum_dag.png'!


In [11]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag

# 1. Загружаем схему
qasm_with_cnot = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];
cx q[0], q[1];
t q[1];
"""

circuit = QuantumCircuit.from_qasm_str(qasm_with_cnot)
dag = circuit_to_dag(circuit)

# 2. Новая функция анализа с поддержкой Qiskit 1.x
def analyze_cnot_connections(dag_circuit):
    print("=== АНАЛИЗ ДВУХКУБИТНЫХ СВЯЗЕЙ В ГРАФЕ ===\n")

    # Создаем карту индексов кубитов прямо из графа
    qubit_map = {qubit: index for index, qubit in enumerate(dag_circuit.qubits)}

    for node in dag_circuit.op_nodes():
        if node.name == 'cx':
            # Извлекаем объекты кубитов
            control_qubit = node.qargs[0]
            target_qubit = node.qargs[1]

            # Находим их порядковые номера через созданную карту
            c_index = qubit_map[control_qubit]
            t_index = qubit_map[target_qubit]

            c_name = f"q[{c_index}]"
            t_name = f"q[{t_index}]"

            print(f" Найдена двухкубитная связь: вентиль [{node.name.upper()}]")
            print(f"   ├─ ВЕДУЩАЯ ветвь (Control): {c_name}")
            print(f"   └─ ВЕДОМАЯ ветвь (Target) : {t_name}")
            print("-" * 40)

# Запускаем анализ
analyze_cnot_connections(dag)


=== АНАЛИЗ ДВУХКУБИТНЫХ СВЯЗЕЙ В ГРАФЕ ===

 Найдена двухкубитная связь: вентиль [CX]
   ├─ ВЕДУЩАЯ ветвь (Control): q[0]
   └─ ВЕДОМАЯ ветвь (Target) : q[1]
----------------------------------------


In [13]:
from qiskit import QuantumCircuit, qasm2
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit.library import TGate
# Импортируем класс операционного узла для проверки типа
from qiskit.dagcircuit import DAGOpNode

qasm_before = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];
cx q[0],q[1];
"""

circuit = QuantumCircuit.from_qasm_str(qasm_before)
dag = circuit_to_dag(circuit)

print("--- СХЕМА ДО ПРОТАСКИВАНИЯ ---")
print(qasm2.dumps(dag_to_circuit(dag)))

def commute_t_forward(dag_circuit):
    for cx_node in list(dag_circuit.op_nodes()):
        if cx_node.name == 'cx':
            control_qubit = cx_node.qargs[0] # Берем ведущий кубит

            for pred in dag_circuit.predecessors(cx_node):
                # Изменение здесь: вместо .type == 'op' проверяем класс узла через isinstance
                if isinstance(pred, DAGOpNode) and control_qubit in pred.qargs and pred.name == 't':

                    print(f"\n[Успех] Найдена возможность комммутации!")
                    print(f"Протаскиваем гейт [{pred.name.upper()}] вперед через ведущую линию CNOT...")

                    # Удаляем старый узел T
                    dag_circuit.remove_op_node(pred)

                    # Вставляем новый T-гейт ПОСЛЕ cx_node на ведущий кубит.
                    # apply_operation_back добавляет операцию в самый конец текущего пути кубита
                    dag_circuit.apply_operation_back(TGate(), qargs=[control_qubit])

                    print("Перестановка успешно выполнена!")
                    return dag_circuit

    return dag_circuit

optimized_dag = commute_t_forward(dag)
optimized_circuit = dag_to_circuit(optimized_dag)

print("\n--- СХЕМА ПОСЛЕ АВТОМАТИЧЕСКОГО ПРОТАСКИВАНИЯ ---")
print(qasm2.dumps(optimized_circuit))


--- СХЕМА ДО ПРОТАСКИВАНИЯ ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
t q[0];
cx q[0],q[1];

[Успех] Найдена возможность комммутации!
Протаскиваем гейт [T] вперед через ведущую линию CNOT...
Перестановка успешно выполнена!

--- СХЕМА ПОСЛЕ АВТОМАТИЧЕСКОГО ПРОТАСКИВАНИЯ ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
cx q[0],q[1];
t q[0];


In [14]:
from qiskit import QuantumCircuit, qasm2
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit.library import TGate, SGate
from qiskit.dagcircuit import DAGOpNode

# 1. Более сложная "грязная" схема: T-гейты разделены CNOT, но стоят на одной ведущей линии!
qasm_complex = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];         // Первый дорогой T-гейт (до CNOT)
cx q[0],q[1];   // CNOT связь
t q[0];         // Второй дорогой T-гейт (после CNOT)
"""

circuit = QuantumCircuit.from_qasm_str(qasm_complex)
dag = circuit_to_dag(circuit)

print("--- ИСХОДНАЯ СХЕМА ---")
print(qasm2.dumps(dag_to_circuit(dag)))

def full_pipeline_optimizer(dag_circuit):
    # --- ШАГ 1: Протаскивание (Коммутация) ---
    for cx_node in list(dag_circuit.op_nodes()):
        if cx_node.name == 'cx':
            control_qubit = cx_node.qargs[0]

            for pred in dag_circuit.predecessors(cx_node):
                if isinstance(pred, DAGOpNode) and control_qubit in pred.qargs and pred.name == 't':
                    print("\n[Оптимизатор]: Найдено скрытое T-звено. Протаскиваем сквозь CNOT...")
                    dag_circuit.remove_op_node(pred)
                    dag_circuit.apply_operation_back(TGate(), qargs=[control_qubit])
                    break # Успешно передвинули, выходим из внутреннего цикла

    # --- ШАГ 2: Поиск n-грамм и схлопывание фаз ---
    # Перестраиваем последовательность узлов после изменения графа
    for wire in dag_circuit.wires:
        nodes = list(dag_circuit.nodes_on_wire(wire))
        i = 0
        while i < len(nodes) - 1:
            node1 = nodes[i]
            node2 = nodes[i+1]

            if isinstance(node1, DAGOpNode) and isinstance(node2, DAGOpNode):
                # Если два T-гейта оказались рядом после переноса:
                if node1.name == 't' and node2.name == 't':
                    print("[Оптимизатор]: Бум! Обнаружена биграмма [T] + [T]. Схлопываем в гейт [S]...")
                    dag_circuit.remove_op_node(node1)
                    dag_circuit.remove_op_node(node2)
                    dag_circuit.apply_operation_back(SGate(), qargs=[wire])
                    break
            i += 1

    return dag_circuit

# Запускаем полный пайплайн
optimized_dag = full_pipeline_optimizer(dag)
optimized_circuit = dag_to_circuit(optimized_dag)

print("\n--- ФИНАЛЬНАЯ ОПТИМИЗИРОВАННАЯ СХЕМА ---")
print(qasm2.dumps(optimized_circuit))


--- ИСХОДНАЯ СХЕМА ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
t q[0];
cx q[0],q[1];
t q[0];

[Оптимизатор]: Найдено скрытое T-звено. Протаскиваем сквозь CNOT...
[Оптимизатор]: Бум! Обнаружена биграмма [T] + [T]. Схлопываем в гейт [S]...

--- ФИНАЛЬНАЯ ОПТИМИЗИРОВАННАЯ СХЕМА ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
cx q[0],q[1];
s q[0];
